## CSV vs Apache Parquet Benchmark

### Objective

Compare the storage efficiency as well as the read and write performance of CSV and Apache Parquet using the same weather dataset.

### Dataset

- Source: NOAA Daily Climate Data
- Number of records: **422.2 million**
- Number of columns: **8**

### File Size Comparison

| Format | File Size |
|---------|----------:|
| CSV | 15.4 GB |
| Apache Parquet | 1.5 GB |

### Write Performance

| Format | Export Time |
|---------|------------:|
| CSV | 9 min 11 s |
| Apache Parquet | 3 min 45 s |

### Read Performance (Full Dataset)
Measure the time required to load each file into a pandas DataFrame.

> **Important**
>
> CSV and Parquet are measured separately to ensure comparable results.

| Format | Read Time |
|---------|----------:|
| CSV | Not completed (MemoryError) |
| Apache Parquet | Not completed (ArrowMemoryError) |

### Read Performance (400 Million Rows)

| Format | Read Time |
|---------|----------:|
| CSV | 8 min 11 s |
| Apache Parquet | 3 min 4 s |

### Summary

Apache Parquet reduced the storage size of the dataset from **15.4 GB** to **1.5 GB** while also significantly improving write performance compared to CSV.

A full read performance benchmark could not be completed because the complete dataset (~422 million observations) exceeded the available system memory (32 GB RAM) when loaded into a pandas DataFrame. However, the benchmark performed on **400 million observations** showed that **Apache Parquet was approximately 2.7× faster than CSV** when reading the dataset.

> **Note:** Benchmark results depend on the hardware, storage device, operating system, and software versions. The measurements presented here are intended for illustrative purposes only.
>
> **Test environment**
> - CPU: Intel® Core™ 7 240H @ 2.50 GHz
> - Memory: 32 GB DDR5 (5600 MT/s)
> - Storage: 1 TB SK hynix PVC10 NVMe SSD
> - Operating System: Windows 11 (64-bit)
> - Python: 3.14
> - Pandas: 3.0.3
> - PyArrow: 25.0.1

### CSV Read Performance

In [1]:
# Import libraries and define paths
from pathlib import Path
import pandas as pd
import time

RAW_DATA_PATH = Path(
    r"C:\Users\zychl\Desktop\Data Engineering\weather_data_processing\00_raw_data"
)

CSV_EXPORT = RAW_DATA_PATH / "weather.csv"
PARQUET_EXPORT = RAW_DATA_PATH / "weather.parquet"

In [ ]:
# Record the current high-resolution timestamp before starting the operation.
# The elapsed execution time can be calculated by subtracting this value
# from another timestamp recorded after the operation completes.
start = time.perf_counter()

weather_csv = pd.read_csv(CSV_EXPORT)

# Record the current timestamp after the operation has finished.
end = time.perf_counter()

print(f'CSV read time: {end - start:.2f} seconds')

### Parquet Read Performance

In [ ]:
start = time.perf_counter()

weather_parquet = pd.read_parquet(PARQUET_EXPORT)

end = time.perf_counter()

print(f'Parquet read time: {end - start:.2f} seconds')

> **Note:**
>
> Due to hardware memory limitations (32 GB RAM), the complete dataset (~422 million observations) could not be fully loaded into a pandas DataFrame for read performance benchmarking. This demonstrates one of the practical limitations of in-memory processing when working with large-scale datasets.
>
> Therefore, the following benchmark was performed using a **400 million row sample** of the dataset.

### Read Performance (400 Million Row Sample)

#### CSV

In [7]:
start = time.perf_counter()
weather_csv = pd.read_csv(
    CSV_EXPORT,
    nrows=400_000_000)
end = time.perf_counter()

print(f'CSV read time: {end - start:.2f} seconds')

CSV read time: 481.19 seconds


In [6]:
del weather_csv

#### Parquet

In [ ]:
# Source - https://stackoverflow.com/a/69888274
# Posted by David Kaftan
# Retrieved 2026-08-29, License - CC BY-SA 4.0

from pyarrow.parquet import ParquetFile
import pyarrow as pa 

start = time.perf_counter()

pf = ParquetFile(PARQUET_EXPORT) 
batch_size = next(pf.iter_batches(batch_size = 400_000_000)) 
df = pa.Table.from_batches([batch_size]).to_pandas() 

end = time.perf_counter()

print(f'Parquet read time: {end - start:.2f} seconds')

Parquet read time: 179.25 seconds
